In [1]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM - SECONDARY DATASET (Olist)
 Phase 2: Data Cleaning -> olist_orders_dataset.csv (for Tableau dashboard)
==================================================================================
 Purpose : Clean the core orders table for use in Tableau. This is the central
           dimension table joined to customers (customer_id), order_items and
           payments (order_id).

 IMPORTANT NOTES FOR A BI/DASHBOARD DATASET:
   1. Missing 'order_approved_at' / 'order_delivered_carrier_date' /
      'order_delivered_customer_date' are EXPECTED for orders that are not
      yet delivered (shipped, processing, canceled, etc.) -> these are NOT
      errors and must not be imputed with fake dates.
   2. However, 8 rows have order_status = 'delivered' but a missing
      order_delivered_customer_date -> this IS a genuine data inconsistency
      and is flagged explicitly rather than silently left unnoticed.
   3. No rows are dropped: this table's row count must match the number of
      unique orders referenced by order_items and payments, or those tables'
      joins will show missing/blank order-level data in Tableau.
==================================================================================
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. LOAD RAW DATA
# ----------------------------------------------------------------------------------
section("1. LOAD RAW DATA")

RAW_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/secondary Data/olist_orders_dataset.csv"
CLEANED_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/cleaned/olist_orders_cleaned.csv"

df = pd.read_csv(RAW_PATH)
print(f"Raw dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")


# ----------------------------------------------------------------------------------
# 2. DUPLICATE RECORD CHECK
# ----------------------------------------------------------------------------------
section("2. DUPLICATE RECORD CHECK")

full_dupes = df.duplicated().sum()
id_dupes = df["order_id"].duplicated().sum()

print(f"Fully duplicated rows      : {full_dupes}")
print(f"Duplicated order_id count  : {id_dupes}")

before_rows = df.shape[0]
df = df.drop_duplicates()   # safety net; confirmed 0 found
after_rows = df.shape[0]
print(f"Rows removed as duplicates  : {before_rows - after_rows}")


# ----------------------------------------------------------------------------------
# 3. DATA TYPE CORRECTION (date parsing)
# ----------------------------------------------------------------------------------
section("3. DATA TYPE CORRECTION")

for col in DATE_COLS:
    df[col] = pd.to_datetime(df[col])
print("All 5 date columns parsed to datetime:")
print(df[DATE_COLS].dtypes)


# ----------------------------------------------------------------------------------
# 4. MISSING VALUE ANALYSIS (expected vs genuine anomaly)
# ----------------------------------------------------------------------------------
section("4. MISSING VALUE ANALYSIS")

missing = df.isnull().sum()
missing = missing[missing > 0]
print("Missing values by column:")
print(missing)

print("\nCross-check: missing 'order_delivered_customer_date' by order_status")
print(df.loc[df["order_delivered_customer_date"].isnull(), "order_status"].value_counts())


# ----------------------------------------------------------------------------------
# 5. GENUINE DATA ANOMALY FLAG (delivered status but no delivery date)
# ----------------------------------------------------------------------------------
section("5. DATA ANOMALY FLAG: 'delivered' STATUS WITH NO DELIVERY DATE")

anomaly_mask = (df["order_status"] == "delivered") & (df["order_delivered_customer_date"].isnull())
anomaly_count = anomaly_mask.sum()

print(f"Orders marked 'delivered' with a missing delivery date: {anomaly_count}")
if anomaly_count > 0:
    print(df.loc[anomaly_mask, ["order_id", "order_status", "order_delivered_customer_date"]])
    print("\nThese dates are NOT fabricated. Left as missing (NaT) and flagged via a new")
    print("'data_quality_flag' column so they can be filtered/highlighted in Tableau")
    print("rather than silently treated as normal missing data.")

df["data_quality_flag"] = np.where(anomaly_mask, "delivered_missing_date", "ok")


# ----------------------------------------------------------------------------------
# 6. ORDER STATUS DISTRIBUTION
# ----------------------------------------------------------------------------------
section("6. ORDER STATUS DISTRIBUTION")
print(df["order_status"].value_counts())


# ----------------------------------------------------------------------------------
# 7. FINAL VALIDATION
# ----------------------------------------------------------------------------------
section("7. FINAL VALIDATION")

print(f"Final shape                       : {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Remaining duplicate rows          : {df.duplicated().sum()}")
print(f"order_id still fully unique       : {df['order_id'].is_unique}")
print(f"Row count unchanged from raw file : {df.shape[0] == before_rows}")


# ----------------------------------------------------------------------------------
# 8. EXPORT CLEANED DATASET
# ----------------------------------------------------------------------------------
section("8. EXPORT CLEANED DATASET")

df.to_csv(CLEANED_PATH, index=False)
print(f"Cleaned dataset saved to: {CLEANED_PATH}")

section("ORDERS DATASET CLEANING COMPLETE")
print("This file is now safe to load into Tableau as the central order dimension,")
print("joined to customers, order_items, and payments on their respective ID columns.")


 1. LOAD RAW DATA
Raw dataset loaded: 99441 rows, 8 columns

 2. DUPLICATE RECORD CHECK
Fully duplicated rows      : 0
Duplicated order_id count  : 0
Rows removed as duplicates  : 0

 3. DATA TYPE CORRECTION
All 5 date columns parsed to datetime:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

 4. MISSING VALUE ANALYSIS
Missing values by column:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Cross-check: missing 'order_delivered_customer_date' by order_status
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

 5. DATA ANOMALY FLAG: 'delivered' STATUS WITH 